# 05 — Labeled Lip Clip Builder

Notebook นี้ต่อจาก `04_speaker_track_alignment.ipynb`

เป้าหมายคือสร้าง **ตัวอย่าง lip-reading ที่มี label จริงในระดับ segment** จาก:
- mouth crops 96×96
- transcript
- candidate speaker / track

Input:
- `dataset/mouth_crops/video001/mouth_metadata.csv`
- `dataset/speaker_alignment/video001/speaker_track_alignment_auto.csv`

Output:
- `dataset/labeled_clips/video001/...`
- `manifest.csv`
- `manifest_train_ready.csv`

หลักการ:
1. ใช้เฉพาะ segment ที่สถานะ `AUTO`
2. เลือก mouth frames ของ track ที่ถูกจับคู่
3. ตรวจ continuity / coverage
4. คัด segment ที่สั้นหรือยาวเกินไป
5. สร้าง clip folder
6. สร้าง manifest สำหรับ train

## Cell 1 — Path

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

MOUTH_META = (
    PROJECT_ROOT
    / "dataset"
    / "mouth_crops"
    / VIDEO_ID
    / "mouth_metadata.csv"
)

ALIGNMENT_CSV = (
    PROJECT_ROOT
    / "dataset"
    / "speaker_alignment"
    / VIDEO_ID
    / "speaker_track_alignment_auto.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "dataset"
    / "labeled_clips"
    / VIDEO_ID
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("MOUTH_META   :", MOUTH_META)
print("ALIGNMENT_CSV:", ALIGNMENT_CSV)
print("OUTPUT_ROOT  :", OUTPUT_ROOT)

if not MOUTH_META.exists():
    raise FileNotFoundError(MOUTH_META)

if not ALIGNMENT_CSV.exists():
    raise FileNotFoundError(ALIGNMENT_CSV)

## Cell 2 — โหลดข้อมูล

In [ ]:
import pandas as pd

mouth_df = pd.read_csv(MOUTH_META)
align_df = pd.read_csv(ALIGNMENT_CSV)

if mouth_df["accepted"].dtype == object:
    mouth_df["accepted"] = (
        mouth_df["accepted"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    )

mouth_df = mouth_df[
    mouth_df["accepted"] == True
].copy()

mouth_df["track_id"] = mouth_df["track_id"].astype(int)
mouth_df["frame_index"] = mouth_df["frame_index"].astype(int)

align_df["candidate_track_id"] = (
    align_df["candidate_track_id"]
    .astype(int)
)

align_df["start_frame"] = align_df["start_frame"].astype(int)
align_df["end_frame"] = align_df["end_frame"].astype(int)

print("Mouth rows:", len(mouth_df))
print("AUTO segments:", len(align_df))

display(mouth_df.head())
display(align_df.head())

## Cell 3 — ตั้งเกณฑ์ QA

ค่าเริ่มต้น:
- อย่างน้อย 8 mouth frames
- coverage อย่างน้อย 85%
- segment ไม่เกิน 8 วินาที
- segment สั้นสุด 0.25 วินาที

ปรับได้ภายหลังตาม dataset จริง

In [ ]:
MIN_FRAMES = 8
MIN_COVERAGE = 0.85
MIN_DURATION_SEC = 0.25
MAX_DURATION_SEC = 8.0

print("MIN_FRAMES       :", MIN_FRAMES)
print("MIN_COVERAGE     :", MIN_COVERAGE)
print("MIN_DURATION_SEC :", MIN_DURATION_SEC)
print("MAX_DURATION_SEC :", MAX_DURATION_SEC)

In [ ]:
print("Alignment rows:", len(align_df))
print("Alignment columns:", align_df.columns.tolist())

display(align_df.head(10))

## Cell 4 — ตรวจแต่ละ Segment

In [ ]:
import numpy as np

qa_rows = []

for _, seg in align_df.iterrows():

    segment_id = int(seg["segment_id"])
    track_id = int(seg["candidate_track_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])
    start_time = float(seg["start_time"])
    end_time = float(seg["end_time"])
    text = str(seg["text"]).strip()

    duration = end_time - start_time
    expected_frames = max(
        1,
        end_frame - start_frame + 1
    )

    matched = mouth_df[
        (mouth_df["track_id"] == track_id)
        &
        (mouth_df["frame_index"] >= start_frame)
        &
        (mouth_df["frame_index"] <= end_frame)
    ].copy()

    matched = matched.sort_values(
        "frame_index"
    )

    observed_frames = int(
        matched["frame_index"].nunique()
    )

    coverage = (
        observed_frames / expected_frames
        if expected_frames > 0 else 0.0
    )

    reasons = []

    if observed_frames < MIN_FRAMES:
        reasons.append("too_few_frames")

    if coverage < MIN_COVERAGE:
        reasons.append("low_coverage")

    if duration < MIN_DURATION_SEC:
        reasons.append("too_short")

    if duration > MAX_DURATION_SEC:
        reasons.append("too_long")

    if not text:
        reasons.append("empty_text")

    status = "ACCEPT" if not reasons else "REJECT"

    qa_rows.append({
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "track_id": track_id,
        "start_time": start_time,
        "end_time": end_time,
        "duration": round(duration, 3),
        "start_frame": start_frame,
        "end_frame": end_frame,
        "expected_frames": expected_frames,
        "observed_frames": observed_frames,
        "coverage": round(coverage, 4),
        "text": text,
        "status": status,
        "reject_reason": "|".join(reasons),
    })

qa_df = pd.DataFrame(qa_rows)

print(qa_df["status"].value_counts())
display(qa_df.head(30))

## Cell 5 — ดูเหตุผลที่ reject

In [ ]:
display(
    qa_df[
        qa_df["status"] == "REJECT"
    ][
        [
            "segment_id",
            "track_id",
            "duration",
            "observed_frames",
            "coverage",
            "text",
            "reject_reason",
        ]
    ].head(50)
)

## Cell 6 — สร้าง Clip Folder สำหรับ segment ที่ ACCEPT

แต่ละ segment จะเก็บ mouth frames ตามลำดับเวลา

In [ ]:
import shutil

ACCEPT_ROOT = OUTPUT_ROOT / "clips"
ACCEPT_ROOT.mkdir(parents=True, exist_ok=True)

manifest_rows = []

accepted_df = qa_df[
    qa_df["status"] == "ACCEPT"
].copy()

for _, seg in accepted_df.iterrows():

    segment_id = int(seg["segment_id"])
    track_id = int(seg["track_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])

    matched = mouth_df[
        (mouth_df["track_id"] == track_id)
        &
        (mouth_df["frame_index"] >= start_frame)
        &
        (mouth_df["frame_index"] <= end_frame)
    ].copy().sort_values("frame_index")

    clip_id = (
        f"{VIDEO_ID}_seg{segment_id:06d}_"
        f"track{track_id:03d}"
    )

    clip_dir = ACCEPT_ROOT / clip_id
    clip_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    copied = 0
    frame_indices = []

    for order_index, (_, row) in enumerate(
        matched.iterrows()
    ):
        src = PROJECT_ROOT / str(
            row["mouth_file"]
        )

        if not src.exists():
            continue

        dst = (
            clip_dir
            / f"{order_index:04d}.jpg"
        )

        shutil.copy2(
            src,
            dst
        )

        copied += 1
        frame_indices.append(
            int(row["frame_index"])
        )

    if copied == 0:
        continue

    manifest_rows.append({
        "clip_id": clip_id,
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "track_id": track_id,
        "clip_path": str(
            clip_dir.relative_to(PROJECT_ROOT)
        ),
        "num_frames": copied,
        "first_frame": min(frame_indices),
        "last_frame": max(frame_indices),
        "start_time": float(seg["start_time"]),
        "end_time": float(seg["end_time"]),
        "duration": float(seg["duration"]),
        "coverage": float(seg["coverage"]),
        "text": seg["text"],
    })

manifest_df = pd.DataFrame(
    manifest_rows
)

print("Clips created:", len(manifest_df))
display(manifest_df.head(20))

## Cell 7 — Export manifest

In [ ]:
MANIFEST_CSV = OUTPUT_ROOT / "manifest.csv"
QA_CSV = OUTPUT_ROOT / "segment_qa.csv"

manifest_df.to_csv(
    MANIFEST_CSV,
    index=False,
    encoding="utf-8-sig"
)

qa_df.to_csv(
    QA_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(" -", MANIFEST_CSV)
print(" -", QA_CSV)

## Cell 8 — เพิ่มสถิติข้อความ

เรายังไม่ tokenize ภาษาไทยในขั้นนี้  
แค่ตรวจความยาวข้อความและจำนวน frame

In [ ]:
manifest_df["text_length"] = (
    manifest_df["text"]
    .astype(str)
    .str.len()
)

summary = manifest_df[
    [
        "num_frames",
        "duration",
        "coverage",
        "text_length",
    ]
].describe()

display(summary)

## Cell 9 — Preview Clip ตัวอย่าง

แสดง 1 clip เป็นภาพเรียงบางเฟรม

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math

if len(manifest_df) == 0:
    raise RuntimeError("ยังไม่มี clip ที่ ACCEPT")

sample = manifest_df.iloc[0]

clip_dir = PROJECT_ROOT / sample["clip_path"]
frames = sorted(
    clip_dir.glob("*.jpg")
)

print("Clip ID :", sample["clip_id"])
print("Text    :", sample["text"])
print("Frames  :", len(frames))

sample_count = min(12, len(frames))

indices = np.linspace(
    0,
    len(frames) - 1,
    sample_count
).astype(int)

fig, axes = plt.subplots(
    3,
    4,
    figsize=(12, 9)
)

axes = axes.flatten()

for ax in axes:
    ax.axis("off")

for ax, idx in zip(
    axes,
    indices
):
    img = cv2.imread(
        str(frames[idx])
    )
    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )
    ax.imshow(img)
    ax.set_title(
        f"{frames[idx].stem}"
    )
    ax.axis("off")

plt.suptitle(
    sample["text"]
)
plt.tight_layout()
plt.show()

## Cell 10 — สร้าง train-ready manifest เบื้องต้น

ตอนนี้ยังไม่ split train/val/test เพราะควร split ตาม speaker/video  
เพื่อป้องกัน data leakage

In [ ]:
TRAIN_READY_CSV = (
    OUTPUT_ROOT
    / "manifest_train_ready.csv"
)

train_ready = manifest_df[
    [
        "clip_id",
        "clip_path",
        "num_frames",
        "track_id",
        "video_id",
        "text",
    ]
].copy()

train_ready["label_status"] = "weak_label"

train_ready.to_csv(
    TRAIN_READY_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", TRAIN_READY_CSV)
display(train_ready.head(20))

# ขั้นต่อไป — 06 Dataset Split + Baseline Training

หลัง Notebook นี้ผ่าน เราจะมี:

```text
dataset/labeled_clips/video001/
├── clips/
│   ├── video001_seg000001_track000/
│   │   ├── 0000.jpg
│   │   ├── 0001.jpg
│   │   └── ...
│   └── ...
├── manifest.csv
├── segment_qa.csv
└── manifest_train_ready.csv
```

ขั้นต่อไป:
1. ทำ video002 แบบเดียวกัน
2. รวม manifest หลายวิดีโอ
3. ตรวจ weak labels
4. split train / val / test แบบ speaker-aware
5. train baseline lip-reading model